## **Summarize Messages**

In [1]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ]
)

In [2]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolois?"),
        AIMessage(content="its grate weather but why you need?"),
        HumanMessage(content="I want to go there so can you tell me!!"),
        AIMessage(content="Why you come hare do you have enough amount of money."),
        HumanMessage(content="I'm a Ai engineer so i have huge money i've 10 taka. do know 10 taka how much value?"),
        AIMessage(content="Ahh 10 take is huge you can round the world using this ig."),
        HumanMessage(content="So tell me what is the current wether of there?")
    ]},
    {"configurable": {"thread_id": "chat_1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\nThe capital of the moon is Lunapolis. The user inquired about the weather in Lunapolis, expressing a desire to visit. They mentioned being an AI engineer and having 10 taka, questioning its value. The AI humorously responded that 10 taka is significant, suggesting it could allow for traveling around the world.', additional_kwargs={}, response_metadata={}, id='535407d7-df53-4b08-8eec-3e6bd3689722'),
              HumanMessage(content='So tell me what is the current wether of there?', additional_kwargs={}, response_metadata={}, id='22cffdc2-7d93-444d-8367-014276140eff'),
              AIMessage(content='There isn’t a “current weather” on the Moon because it has no atmosphere. No wind, clouds, rain, or precipitation exist there. What you can talk about are temperature, sunlight, and the lunar day-night cycle.\n\n- Temperature ranges: daytime surface temps around 127°C (260°F); nighttime temps around −173

## **Trim and delete**

In [3]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage


@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages]
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of moon?"),
        ToolMessage(content="Hello its respone from mocking tool", tool_call_id="2"),
        AIMessage(content="The capital of the moon is Lunapolis."),

        HumanMessage(content="What is the weather in Lunapolois?"),
        AIMessage(content="its grate weather but why you need?"),
        
        HumanMessage(content="I want to go there so can you tell me!!"),
        ToolMessage(content="THis is mocking tool", tool_call_id="1"),
        AIMessage(content="Why you come hare do you have enough amount of money."),
        HumanMessage(content="I'm a Ai engineer so i have huge money i've 10 taka. do know 10 taka how much value?"),
        AIMessage(content="Ahh 10 take is huge you can round the world using this ig."),
        HumanMessage(content="So tell me what is the current wether of there?")
    ]},
    {"configurable": {"thread_id": "chat_2"}}
)

pprint(response)

{'messages': [HumanMessage(content='What is the capital of moon?', additional_kwargs={}, response_metadata={}, id='4a67c353-009c-4c3e-8cb3-a0c15319ab5c'),
              AIMessage(content='The capital of the moon is Lunapolis.', additional_kwargs={}, response_metadata={}, id='9565ab09-86ea-4f15-9ae6-eb6dc371a07a'),
              HumanMessage(content='What is the weather in Lunapolois?', additional_kwargs={}, response_metadata={}, id='f9d222a2-6aa4-408e-ae71-7b720503fc16'),
              AIMessage(content='its grate weather but why you need?', additional_kwargs={}, response_metadata={}, id='5e2dfe61-caac-4310-a509-226203c4cdb0'),
              HumanMessage(content='I want to go there so can you tell me!!', additional_kwargs={}, response_metadata={}, id='c9baeb66-cfaa-4df2-ab76-71a80521a879'),
              AIMessage(content='Why you come hare do you have enough amount of money.', additional_kwargs={}, response_metadata={}, id='a2d79566-9629-4f52-9668-89d91892e7c4'),
              HumanMe